First, we will import our packages (numpy and gurobipy) as well as initialize (make up) the data for our problem.

In [1]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

## First Stage Constants
i = np.array([6,14,27])
s = np.array([1,5,3])
B = 1000
S = 100
## Second Stage Constants
K = 50 # Number of scenarios
d_xi = np.array([
    np.random.normal(70,10,K),
    np.random.normal(8,2,K),
    np.random.normal(15,4,K)
])
c = np.array([7,18,30])
p = np.array([1.0/K]*K) # Probability of a scenario happening

In this problem i is the investment cost, sis the space each item takes up, Bis our budget, and Sis our maximum space. Another variable of note is the demand in each scenario d_xi. We can then use Gurobi to solve our problem. First, we will define our model with gp.model(), set the goal (minimize) as well as initialize our decision variables.

In [2]:
m = gp.Model("2SP")
m.ModelSense = GRB.MINIMIZE
m.setParam('OutputFlag', 0)  # Telling gurobi to not be verbose
m.params.logtoconsole=0
# VARIABLES 
## First Stage Decision variable
x = m.addMVar((3,), vtype = GRB.INTEGER, name = "x")

## 2nd stage decision for each potential scenario
y = m.addMVar((3,K), vtype = GRB.INTEGER, name = "y")
### Gurobi automatically sets bounds to (0, inf)

Set parameter Username
Academic license - for non-commercial use only - expires 2026-05-29


As you can see in the above examples, both our demand (d_xi) and our second-stage decision variable (y) are not single-dimensional vectors, but rather a matrix since both of these have a scenario index in our formulation. After this we can define our objective function:

In [3]:
# OBJECTIVE 
m.setObjective(i @ x + gp.quicksum(c @ y[:,k] * p[k] for k in range(K)))

One nice thing about Gurobi is that it supports numpy matrix multiplication operations when defining constraints and objectives, making it simple to code if you think in matrix multiplications (ex. i @ x). Gurobi also supports summation operations, making the barrier to entry lower as well. Here we calculate the first stage cost, then take the average second stage cost.
We can now move on to defining our constraints:

In [4]:
# CONSTRAINTS 
## Installation Budget
m.addConstr(i @ x <= B)

## Space
m.addConstr(s @ x <= S)

## Meet demand in each scenario
m.addConstrs((
    x + y[:,k] >= d_xi[:,k] for k in range(K)
))

{0: <MConstr (3,) *awaiting model update*>,
 1: <MConstr (3,) *awaiting model update*>,
 2: <MConstr (3,) *awaiting model update*>,
 3: <MConstr (3,) *awaiting model update*>,
 4: <MConstr (3,) *awaiting model update*>,
 5: <MConstr (3,) *awaiting model update*>,
 6: <MConstr (3,) *awaiting model update*>,
 7: <MConstr (3,) *awaiting model update*>,
 8: <MConstr (3,) *awaiting model update*>,
 9: <MConstr (3,) *awaiting model update*>,
 10: <MConstr (3,) *awaiting model update*>,
 11: <MConstr (3,) *awaiting model update*>,
 12: <MConstr (3,) *awaiting model update*>,
 13: <MConstr (3,) *awaiting model update*>,
 14: <MConstr (3,) *awaiting model update*>,
 15: <MConstr (3,) *awaiting model update*>,
 16: <MConstr (3,) *awaiting model update*>,
 17: <MConstr (3,) *awaiting model update*>,
 18: <MConstr (3,) *awaiting model update*>,
 19: <MConstr (3,) *awaiting model update*>,
 20: <MConstr (3,) *awaiting model update*>,
 21: <MConstr (3,) *awaiting model update*>,
 22: <MConstr (3,) *

The first line of code ensures that the cost of installing in-house servers does not exceed our budget B. The second line ensures that we only buy as many servers as we have space for. The third line of code ensures the number of installed servers and amount of cloud compute resources in scenario k leveraged are enough to meet the demand in scenario k.

Now that the problem is fully defined, we can simply solve the problem and get our optimal decision by calling m.optimize():



In [5]:
# SOLVING
m.optimize()
result = x.x

print(f"Our Server farm will cost ${np.round(m.ObjVal,2)}. We should buy {result[0]} CPUs, {result[1]} GPUs, and {result[2]} TPUs ")
# Our Server farm will cost $995.6. We should buy 51.0 CPUs, 5.0 GPUs, and 8.0 TPUs

Our Server farm will cost $1019.64. We should buy 56.0 CPUs, 4.0 GPUs, and 8.0 TPUs 


In [6]:
import sys
print(sys.executable)


c:\Users\betsie_0410\AppData\Local\anaconda3\envs\energiaenv\python.exe
